# Trabajo final · Data Analyst · Norte Seguro

**Seguros de auto y de hogar** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real de ARCA.

## La pregunta oficial

> **¿Por qué cayó la renovación de pólizas en el segundo semestre?**

**Decisión que habilita:** Qué política de aumentos y qué canal priorizar en las renovaciones.

Antes de analizar, **verificá si los datos sostienen la premisa de la pregunta**. A veces no la sostienen, y decirlo es un hallazgo.

Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Analyst · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 2 | SQL y PL/SQL |
| 3 | Estadística |
| 4 | Python: limpieza |
| 5 | Proceso de análisis |
| 6 | Superset |
| 7 | Analítica de negocio |
| 8 | Bitácora y entrega |

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · La base de Norte Seguro

La primera celda **genera la base** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda la carga en SQLite y te deja tres herramientas: `sql(...)` para consultar, `esquema()` para ver las tablas y `anotar(...)` para dejar registrado cada problema de calidad que encuentres.

In [ ]:
import zlib, base64
_BLOB = (
    "eNqlVttu2zgQffdXEAIKkFuZkORL3aTaly76tEgLtC+7hmEwEuUolUSBktykQf59z1CyLTvpYoE1DFviZWbOmZlD5mVtbMuq"
    "rqwfmWpYVfusVlWKR3zrdJLqjG2bLskNtz5rtM21z/bK5qpqdYPF4mrC8GlY3M/KxNSPXEjVtI+15ub2XietuGYlFlhpYduU"
    "vNAVb4RgH1jd716XG8yvj4b53mdWsMxYtmd55RZsrpnVbWfxNunjKtVj12x1UyuKz225OqxZ72VhfmjLBeKVXV0Pj+wt85jn"
    "0w8e95u1lTk87rRteOCzudj0xlO13FbgRm8bveus4Y3WabwMlgNgi3iregAksUN1Rbu11c4tBOAbLFgEQXDNkiLHc53KP1Sr"
    "PllVav7k5ekW4xquvSu2zryPfz7lV8EiffYc6pxQw/hO8xuxQbg6VSlWwiW21dxKBFeqgs8jn4Vzn90AXLjy2WohpDVdlRJa"
    "76epFHZZmdyZPNF87d0QJsL/Eb6toaevnaW/z7rBDHzdIK3xWkYLn8kZ/UTwIcPVRjwDV20KShWy8R3/gSPjRcQ9R4ep2pqU"
    "ZuFeda3zeWd2ynoblmenqqB6gFemi0az9Sjmi019eAuKbL7YiM3IG33yaqD7W14Ckipr7kVBNJ8GIb6eQNqHyVQXreKpemxi"
    "1AA/r4TFPBACeBNVqcIV7zGeW2u+a8fZD31Lf2qnqyRXx9ACYm5GpL3bwEQG7LGL6+1QBvpzljW65Y9a2SYOseZWAXTM5hFK"
    "JiBaHGlxzHr0PSnhKnAVRcWAxagESY/rfHPGQG3zUlHIrg6c5d8YD4lbGOYXlkF+b/ADixZicCSwIyuMIlYKsxuKjXCFkUBl"
    "TSPEnFcJvBzMDkTFPSvOLAH//ZVcLKbBO5eLwV3v6UVty2AJh0Ek3O/gm+rubczCc8Rbq4njp0NurphcRaPM0HswJMzNPa9d"
    "vCjjw165WP5fHM7UeWCmkArigzzwzPvy99P3vsl9l7yTCCCFvku5T3Uim9ZmLVxx781f0zfl9E3qgQSE8YsZl3G/r1V/yDuy"
    "47OZIIOUxVGTuTAFgSVgH14FFkbTGTWLw4XMVKoSYjJAupQzDMG5KbqyauI1aRtG8p+KemMkdHgjiF3St3Omkzu1BVyo9+kd"
    "IfULgWirqk4V9OqQuVbrStgyW5NZneSp2wg0Zq9S9B+q497sKTzwXmoLMYJzn5kqHgeCZY3rSZKx+zMZ2/rsgbQKZmTeamvN"
    "j4aP9QyruVz1Jf8gD3gu+ulBXnYUlzPX1q9ucStkGFB1V05rapM3jal4Js6UdHsS2epC9u5dV9AZSU35snfDmZz9ewBhJKnf"
    "3lMUsHI3PrGdNocRadmluD7IcSL/k7xCumdLktczBEjJqFe+3jzdH3rlQR5LirrgF03Qlz0id/Lks0A6xh0UB/C4YCRuXZWD"
    "2JIT9HBQNldSgPRTob7OTXi12ilXdUfNGqyOqJ5JEsp3wPfyfHu/Wow1b0QKlN+nG4M48NKX6FmjYeii0TCSIxH9SX7WeC4r"
    "9FAaahhgKFTZxz4MHbF4lMr+Kc1Vg7UNfCCblWspaqbCJKcjme5vGARREWK2ui5UouNPCrjERe/SvS64PolGYqoEuNdOMshw"
    "gxoqNJ+TIcfTFrG0OnbXKIhiTqxqVFeqH+JvttPisCWzKonD17ZJANBtv4en1tT9RsfqrTs6z6CAQ4KyuITiVMIhx6Yjo4SI"
    "NHI2DZbTcOG5C966v2rR3Om6TBJPw/7FVZWOs5k4XmefvEGXGhxNTq68Pov07njyjlmmIbw8TyY42LRVdA29vKpO/gGnc5E5"
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import sqlite3, matplotlib.pyplot as plt
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); tablas = {k: v for k, v in t.items() if not k.startswith("_")}
con = sqlite3.connect(":memory:")
for nombre, df in tablas.items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False)

def sql(consulta, parametros=None):
    """Corre una consulta SQL sobre la base y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, con, params=parametros)

def esquema():
    """Lista las tablas, sus columnas y sus filas."""
    for nombre, df in tablas.items(): print(f"{nombre}  ({len(df):,} filas)".replace(",", "."), "\n   ", ", ".join(df.columns))

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset"):
    """Guarda la base como archivo SQLite y un CSV por tabla, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); salida = sqlite3.connect(f"{carpeta}/base.db")
    for nombre, df in tablas.items(): df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, salida, index=False, if_exists="replace")
    salida.close(); print(f"Listo: {carpeta}/base.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta
import os
esquema()

Los datos tienen los problemas de un dato real. Encontrarlos, decidir qué hacer con cada uno y dejarlo escrito es parte de la evidencia.

## 2 · SQL y PL/SQL

Tres consultas. Guardalas en esta sección: son la evidencia del módulo.

1. La renovación por semestre y por canal (`polizas`).
2. La siniestralidad por producto y por tramo de edad (JOIN de `siniestros`, `polizas` y `clientes`).
3. Un control de integridad: ¿hay siniestros anteriores al inicio de la póliza?

SQLite no tiene PL/SQL. Si querés encapsular una lógica, escribila como una función de Python que ejecuta SQL.

In [ ]:
sql("""
-- Consulta 1
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 2
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 3 (control de integridad)
SELECT 1
""")

## 3 · Estadística

Para la variable principal (`monto_reclamado`): **media, mediana, dispersión y distribución**. Decidí cuál resume mejor y escribí por qué.

In [ ]:
# Tu análisis estadístico

## 4 · Python: limpieza

Traé las tablas a pandas (`df = sql("SELECT * FROM tabla")`) y documentá la calidad: **faltantes, duplicados, valores imposibles y categorías inconsistentes**. Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)`.

In [ ]:
# Tu limpieza
# anotar("problema", "tabla.columna", 0, "qué decidimos y por qué")

In [ ]:
bitacora_calidad

## 5 · Proceso de análisis

**Verificá la premisa** de la pregunta oficial antes de seguir. Después armá al menos dos variables nuevas (semestre de vencimiento, edad del cliente, aumento ofrecido, producto).

In [ ]:
# Tu verificación de la premisa y tus variables nuevas

## 6 · Dashboard en Superset

Exportá la base y conectá Apache Superset al archivo. Armá **4 a 6 gráficos, uno por pregunta**, con un orden de lectura y las definiciones a la vista. Si no podés usar Superset, trabajá con gráficos de este notebook.

In [ ]:
exportar_superset()

In [ ]:
# Si no usás Superset, tus gráficos con matplotlib

## 7 · Analítica de negocio

**KPIs** con su denominador: tasa de renovación, siniestralidad (monto pagado sobre prima), frecuencia de siniestros, días de resolución.

**Segmentación:** canal, producto, edad y semestre.

Un **insight** relevante, una **limitación** y una **recomendación** proporcional a la evidencia.

In [ ]:
# Tus KPIs y tu segmentación

## 8 · Bitácora y entrega

Completá esta bitácora editando la celda. Con ella, otra persona tiene que poder rehacer tus números.

**Equipo:** ____  **Integrantes y roles:** ____

**Pregunta y premisa:** ¿los datos sostienen la premisa? ____

| KPI | Fórmula | Universo | Período | Estados o faltantes |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Hechos** (lo que muestran los datos): ____

**Inferencias** (lo que creemos que significa): ____

**Lo que contradice nuestra hipótesis:** ____

**Recomendación** (proporcional a la evidencia): ____

**Límites y siguiente análisis:** ____

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Mi pregunta termina en una decisión y nombra quién la toma.
- [ ] Verifiqué la premisa de la pregunta.
- [ ] Cada KPI declara fórmula, universo, período y tratamiento de estados.
- [ ] Ningún porcentaje se muestra sin decir sobre cuántos casos.
- [ ] La limpieza está anotada: qué encontré, qué hice y cuántas filas afectó.
- [ ] Los hechos están separados de las inferencias y no prometo causalidad.
- [ ] Mostré al menos un resultado que no me conviene.
- [ ] Hay una limitación y un siguiente análisis.